# E8 — Статистическая достоверность (валидность всех Г)

Источник дисперсии — **сид в сборе train** (суррогат переобучается на каждом сиде; так уже сделано в E3–E7), а не seed прогона — это чинит претензию протокола к старому 07_multi_seed (там модели по сидам были идентичны). Сводим значимость по всем гипотезам: Wilcoxon знаковых рангов + поправка Холма + bootstrap CI + размер эффекта (Cohen d), и boxplot'ы по сидам.

In [1]:
import os, sys, glob, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
sys.path.insert(0, os.path.abspath("."))
import article_experiment_utils as U
import protocol_config as P
FAST_MODE = os.environ.get("ARTICLE_FAST", "1") == "1"
pc = P.DEFAULT.resolved(FAST_MODE)
RES = U.results_dir()
def load(pat):
    fs = [pd.read_csv(p) for p in sorted(glob.glob(str(RES / "tables" / pat))) if os.path.getsize(p) > 0]
    return pd.concat(fs, ignore_index=True) if fs else pd.DataFrame()
print("FAST_MODE", FAST_MODE)

FAST_MODE True


## Валидность источника дисперсии (модели по сидам различны)

In [2]:
# Validity of the variance source: models are retrained per TRAIN seed, so they differ
# (unlike the legacy 07_multi_seed where per-seed models were identical -> zero variance).
nd = 7 if FAST_MODE else 21
coefs = []
for s in (0, 1, 2):
    tr = U.aggregate_trajectories([U.collect_rule_based_dataset(pc.cfg_for({"year": y, "start_date": f"{y}-03-01", "n_days": nd}, seed=s), n_days=nd, prbs_scale=0.3) for y in (2018, 2019)], pc.base_cfg(nd))
    b = U.fit_sindy(tr, period=float(pc.period), feature_variant="physics_no_cross", library_degree=1, optimizer="stlsq", denoise="none")
    coefs.append(np.asarray(b.model.coefficients()).ravel())
cv = float(np.array(coefs).std(0).mean())
print(f"cross-seed coefficient std (mean) = {cv:.4g}")
print("per-seed surrogates differ ->", cv > 1e-6, "  (valid variance source; fixes the legacy 07 critique)")

cross-seed coefficient std (mean) = 0.03939
per-seed surrogates differ -> True   (valid variance source; fixes the legacy 07 critique)


## Г1/Г3 — замкнутый EPI (E3)

In [3]:
# H1/H3 (E3): closed-loop EPI, paired by seed -- Wilcoxon + Holm + bootstrap CI + Cohen d.
e3 = pd.read_csv(RES / "tables" / "e3_seeded.csv")
st_rb = U.paired_stats(e3, "epi", baseline="rule_based"); U.save_table(st_rb, RES / "tables" / "e8_stats_vs_rule_based.csv")
st_sd = U.paired_stats(e3, "epi", baseline="sindy_mpc"); U.save_table(st_sd, RES / "tables" / "e8_stats_vs_sindy.csv")
print("vs rule_based:"); display(st_rb.round(3))
print("proposed sindy_mpc vs others:"); display(st_sd.round(3))

vs rule_based:


,method,baseline,metric,n_pairs,mean_diff,ci_low,ci_high,cohen_d,p_value,p_holm,significant
0,grey_box_mpc,rule_based,epi,10,-0.804,-1.252,-0.366,-1.050,0.014,0.055,False
1,nn_mpc,rule_based,epi,10,-6.675,-8.683,-4.818,-2.023,0.002,0.010,True
2,oracle_mpc,rule_based,epi,2,-1.951,-1.965,-1.936,-95.290,0.500,0.500,False
3,ppo,rule_based,epi,3,-17.668,-21.269,-15.866,-5.665,0.250,0.500,False
4,sindy_mpc,rule_based,epi,10,-3.873,-6.708,-0.778,-0.775,0.049,0.146,False


proposed sindy_mpc vs others:


,method,baseline,metric,n_pairs,mean_diff,ci_low,ci_high,cohen_d,p_value,p_holm,significant
0,grey_box_mpc,sindy_mpc,epi,10,3.069,0.211,5.844,0.623,0.049,0.244,False
1,nn_mpc,sindy_mpc,epi,10,-2.802,-7.362,1.179,-0.380,0.557,1.000,False
2,oracle_mpc,sindy_mpc,epi,2,6.540,5.680,7.400,5.379,0.500,1.000,False
3,ppo,sindy_mpc,epi,3,-9.744,-11.904,-8.252,-5.087,0.250,0.750,False
4,rule_based,sindy_mpc,epi,10,3.873,0.778,6.708,0.775,0.049,0.244,False


## Г4а — онлайн-адаптация (E4)

In [4]:
# H4a (E4): adaptation vs static offline, paired by (shift, seed).
e4 = load("e4_seeded_*.csv")
st_e4 = pd.DataFrame()
if not e4.empty:
    dag = e4[e4.method == "dagger"]
    fin = dag.loc[dag.groupby(["shift", "seed"])["dagger_iter"].idxmax()].assign(method="dagger_final")
    e4m = pd.concat([e4[e4.method.isin(["offline", "ekf_sindy"])], fin], ignore_index=True)
    e4m["seed"] = e4m["shift"].astype(str) + "_" + e4m["seed"].astype(str)   # pairing unit
    st_e4 = U.paired_stats(e4m, "epi", baseline="offline", methods=["dagger_final", "ekf_sindy"])
    U.save_table(st_e4, RES / "tables" / "e8_stats_adaptation.csv")
    print("H4a — adaptation vs offline (paired by shift x seed):"); display(st_e4.round(3))

H4a — adaptation vs offline (paired by shift x seed):


,method,baseline,metric,n_pairs,mean_diff,ci_low,ci_high,cohen_d,p_value,p_holm,significant
0,dagger_final,offline,epi,9,1.057,0.777,1.327,2.348,0.004,0.008,True
1,ekf_sindy,offline,epi,9,0.272,-0.605,1.091,0.197,0.496,0.496,False


## Г4в / безопасность — guard (E5) и супервизор отказов (E7)

In [5]:
# H4c (E5 guard) and safety (E7 supervisor): paired violation reduction.
def paired(a, b, label):
    a = np.asarray(a, float); b = np.asarray(b, float); m = ~(np.isnan(a) | np.isnan(b)); a, b = a[m], b[m]
    diff = a - b
    try:
        _, p = wilcoxon(a, b)
    except Exception:
        p = float("nan")
    rng = np.random.default_rng(0); boot = np.array([rng.choice(diff, len(diff), replace=True).mean() for _ in range(2000)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {"comparison": label, "n": int(len(diff)), "mean_reduction": float(diff.mean()),
            "ci_low": float(lo), "ci_high": float(hi), "p_value": float(p), "significant": bool(p < 0.05)}
res = []
e5 = load("e5_grid_*.csv")
if not e5.empty:
    res.append(paired(e5.viol_unguarded, e5.viol_guarded, "E5 guard: violations unguarded->guarded"))
e7 = load("e7_faults_*.csv")
if not e7.empty:
    piv = e7[e7.fault != "none"].pivot_table(index=["fault", "seed"], columns="supervised", values="viol")
    if 0 in piv.columns and 1 in piv.columns:
        res.append(paired(piv[0], piv[1], "E7 supervisor: violations unsup->sup"))
safety = pd.DataFrame(res); U.save_table(safety, RES / "tables" / "e8_stats_safety.csv")
print("H4c / safety — paired violation reduction (Wilcoxon + bootstrap CI):"); display(safety.round(3))

H4c / safety — paired violation reduction (Wilcoxon + bootstrap CI):


,comparison,n,mean_reduction,ci_low,ci_high,p_value,significant
0,E5 guard: violations unguarded->guarded,20,710.700,348.668,1059.055,0.003,True
1,E7 supervisor: violations unsup->sup,18,2037.333,1628.993,2465.065,0.000,True


## Boxplot'ы по сидам

In [6]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
order = [m for m in ["rule_based", "grey_box_mpc", "oracle_mpc", "sindy_mpc", "nn_mpc", "ppo"] if m in e3.method.values]
ax[0].boxplot([e3[e3.method == m]["epi"] for m in order], labels=order)
ax[0].set_title("E3 EPI by controller"); ax[0].axhline(0, color="k", lw=.5); ax[0].tick_params(axis="x", rotation=25)
if not e4.empty:
    o4 = [m for m in ["offline", "ekf_sindy", "dagger_final"] if m in e4m.method.values]
    ax[1].boxplot([e4m[e4m.method == m]["epi"] for m in o4], labels=o4); ax[1].set_title("E4 EPI (adaptation)"); ax[1].tick_params(axis="x", rotation=25)
if not e7.empty and 0 in piv.columns:
    ax[2].boxplot([piv[0].dropna(), piv[1].dropna()], labels=["no supervisor", "supervisor"]); ax[2].set_title("E7 violations")
U.save_figure(fig, RES / "figures" / "e8_boxplots.png"); plt.close(fig); print("saved e8_boxplots.png")

saved e8_boxplots.png


**Итог E8.** Сведена статистика по всем гипотезам с пересбором train по сидам (валидный источник дисперсии). Главные выводы: nn_mpc значимо хуже rule_based; sindy_mpc положителен, но статистически не превосходит сильные эталоны (большой разброс); адаптация (E4) и guard/супервизор (E5/E7) дают значимое снижение нарушений / восстановление EPI. Таблицы: e8_stats_*.csv; рисунок e8_boxplots.png.